In [13]:
using Luxor
using Colors
using Plots
using IterTools
using DataFrames
using OpenStreetMapX
using LightOSM
using KernelDensity
using Parsers
using Downloads
using OSMToolset
using Profile
using ProfileView
using KernelDensity
include("kernel_density.jl")
include("distance.jl")
include("prepare_data.jl")
include("analyse.jl")
include("plots.jl")
include("transform.jl")
include("tile_regression.jl")
include("kernel_density.jl")

kernel_density (generic function with 1 method)

In [ ]:
#vals, area, density, xs, ys = calc_all_tiles_density(prsd,city_centre,road_types,tiles,ncols,nrows)

([927.9611155437888, 7247.4998390818855, 29177.20809950671, 4895.769202058408, 10949.68355071707, 13128.923841272956, 9784.373766161196, 17574.90931331548, 9051.176966705943, 6759.653092715243  …  11333.110922513535, 4334.133728731543, 0.0, 0.0, 88.35158851521518, 14120.25298258539, 30696.872008759896, 12469.055880032949, 12598.909221123993, 2922.3459246069756], [2.007155087218494e6, 2.006638142331966e6, 2.0061210124701846e6, 2.0056036977340684e6, 2.0050861982125505e6, 2.0045685139968176e6, 2.0040506451795204e6, 2.0035325918547467e6, 2.0030143541141634e6, 2.0024959320526132e6  …  2.0071586817262932e6, 2.0066417362258246e6, 2.006124605750757e6, 2.0056072903990536e6, 2.005089790264113e6, 2.0045721054327656e6, 2.0040542359992834e6, 2.0035361820586822e6, 2.003017943701115e6, 2.0024995210222835e6], [4.63398826920013e-6 4.632794780687335e-6 … 4.6244367030806926e-6 4.62323980273777e-6; 3.619206524149931e-5 3.6182743937341695e-5 … 3.611746623863804e-5 3.610811828763295e-5; … ; 6.29155646259953

In [36]:
cities = ["Poznan"]#,"Warsaw","Berlin","Dublin"]

1-element Vector{String}:
 "Poznan"

In [37]:
scrape_config = "poi_config_test.csv"
scr = OSMToolset.ScrapePOIConfig(DataFrame(CSV.File(scrape_config)))
DATA_PATH = "data"
search_area = 1000
attr = :education
wilderness_distance = 300
shape = "rectangle"
calculate_percent = true
num_of_points = 30
distance_sectors = 200.0
num_of_sectors = 100
admin_level = ""
road_types = ["motorway", "trunk", "primary", "secondary", 
                "tertiary", "residential", "service", "living_street",
                "motorway_link", "trunk_link", "primary_link", "secondary_link", 
                "tertiary_link"]    

summaries = []

for city in cities
    data = prepare_city_map(city, #city_name
                admin_level, #admin_level
                search_area, #search_area
                wilderness_distance, #wilderness_distance
                shape, #shap;
                distance_sectors=300,
                rectangle_boundaries= get_city_bounds(city),
                #calculate_percent = calculate_percent,
                #num_of_points = num_of_points,
                scrape_config = scrape_config,
                in_admin_bounds=false,dir=DATA_PATH)

    city_centre = data[2]
    prsd = OpenStreetMapX.parseOSM("data/$city.osm")
    for num in 10:11
        ncols=num
        nrows=num
        tiles = generate_tiles(city,admin_level,ncols,nrows,dir="data")
        tls,xs,ys = calc_all_tiles_length(prsd,city_centre,road_types,tiles,ncols,nrows)
        tls = tls[:,1]
        center_tile = center_in_tile(tiles,city_centre)
        center_density = tls[center_tile]
        percentile = mean(tls .<= center_density)
        summary = [city, num, percentile]
        push!(summaries,summary)
    end
end

┌ Info: Saved map data to cache data/Poznan.osm.cache
└ @ OpenStreetMapX /home/adamkas/.julia/packages/OpenStreetMapX/gCd33/src/parseMap.jl:110


In [38]:
summaries

2-element Vector{Any}:
 Any["Poznan", 10, 0.99]
 Any["Poznan", 11, 0.9504132231404959]

In [16]:
df = DataFrame(mapreduce(permutedims, vcat, summaries), [:city,:ncols,:cdf])

Row,city,ncols,cdf
,Any,Any,Any
1,Kielce,10,1.0
2,Kielce,11,1.0


In [ ]:
#CSV.write("paris_centers.csv",df)

"paris_centers.csv"